# Fine-tuned Chatbot Tim Legal berbasis RAG

---
## 1. Setup & Instalasi Dependensi

In [1]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps unsloth
!pip install trl transformers datasets accelerate bitsandbytes peft rouge-score evaluate


In [2]:
import torch
import os
import re
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Memuat Hugging Face Token dari Kaggle Secrets
try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("Berhasil login ke Hugging Face!")
except Exception as e:
    print(f"Kaggle secrets: {e}")

HF_USERNAME = "afihsan"
REPO_NAME_SFT = f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-finetuned"
REPO_NAME_GRPO = f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-grpo"

print(f"Base SFT Model : https://huggingface.co/{REPO_NAME_SFT}")
print(f"Target GRPO Model: https://huggingface.co/{REPO_NAME_GRPO}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Berhasil login ke Hugging Face!
Base SFT Model : https://huggingface.co/afihsan/qwen2.5-1.5b-legal-chatbot-finetuned
Target GRPO Model: https://huggingface.co/afihsan/qwen2.5-1.5b-legal-chatbot-grpo


---
## 2. Memuat Base Instruct Model

In [3]:
from unsloth import FastLanguageModel

print(f"Memuat model instruct hasil SFT dari {REPO_NAME_SFT}...")
model_grpo, tokenizer_grpo = FastLanguageModel.from_pretrained(
    model_name=REPO_NAME_SFT,
    max_seq_length=2048,
    load_in_4bit=True,
)

# Inisialisasi adapter LoRA untuk proses pelatihan GRPO
model_grpo = FastLanguageModel.get_peft_model(
    model_grpo,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model instruct berhasil dimuat dan siap untuk GRPO!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Memuat model instruct hasil SFT dari afihsan/qwen2.5-1.5b-legal-chatbot-finetuned...
==((====))==  Unsloth 2026.10.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.10.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model instruct berhasil dimuat dan siap untuk GRPO!


---
## 3. Definisi 4 Reward Functions
Empat fungsi reward wajib sesuai rubrik:
1. `format_reward_func`: Reward shaping bertahap tag `<think>` (+0.2), `</think>` (+0.3), format sempurna (+1.0), penalti -0.5 jika duplikat tag.
2. `reasoning_length_reward`: Proporsional 4 tingkat panjang karakter isi `<think>` (0: 0.0; <50: +0.2; 50-199: +0.5; ≥200: +1.0).
3. `correctness_reward`: +1.0 jika output akhir sesuai dengan ground truth output dataset.
4. `language_reward_func`: +1.0 jika bahasa Indonesia murni, penalti -0.5 jika bahasa Inggris.

In [4]:
from rouge_score import rouge_scorer
import re

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def format_reward_func(completions, **kwargs):
    rewards = []
    for text in completions:
        has_start = "<think>" in text
        has_end = "</think>" in text
        
        # Cek format diawali <think>, proses pemikiran ditutup </think>, lalu ada konten jawaban akhir
        is_perfect = bool(re.search(r"^\s*<think>[\s\S]+?</think>\s*\S+", text.strip()))
        
        if is_perfect:
            # OVERRIDE langsung menjadi tepat 1.0 (sesuai ketentuan rubrik, poin maksimal adalah +1.0)
            score = 1.0
        else:
            # Poin bertahap jika belum sempurna
            score = 0.0
            if has_start:
                score += 0.2
            if has_end:
                score += 0.3
                
        # Penalti jika duplikasi tag lebih dari satu kali
        if text.count("<think>") > 1 or text.count("</think>") > 1:
            score -= 0.5
            
        # Pastikan batas atas poin maksimal adalah +1.0
        score = min(score, 1.0)
        rewards.append(score)
    return rewards

def reasoning_length_reward(completions, **kwargs):
    rewards = []
    for text in completions:
        match = re.search(r"<think>([\s\S]*?)(?:</think>|$)", text)
        if not match:
            rewards.append(0.0)
            continue
        
        thought = match.group(1).strip()
        char_count = len(thought)
        
        if char_count == 0:
            rewards.append(0.0)
        elif char_count < 50:
            rewards.append(0.2)
        elif 50 <= char_count < 200:
            rewards.append(0.5)
        else:
            rewards.append(1.0)
    return rewards

def correctness_reward(prompts, completions, answer, **kwargs):
    rewards = []
    for comp, gt in zip(completions, answer):
        final_answer = comp.split("</think>")[-1].strip() if "</think>" in comp else comp.strip()
        gt_clean = str(gt).strip().lower()
        pred_clean = final_answer.lower()
        
        if not pred_clean or not gt_clean:
            rewards.append(0.0)
            continue
            
        # Hitung metrik standar ROUGE-L
        try:
            scores = scorer.score(gt_clean, pred_clean)
            rouge_l = scores['rougeL'].fmeasure
        except Exception:
            # Fallback jika terjadi kendala pada tokenizer rouge
            gt_words = set(re.findall(r"\b\w+\b", gt_clean))
            pred_words = set(re.findall(r"\b\w+\b", pred_clean))
            rouge_l = len(gt_words.intersection(pred_words)) / max(len(gt_words), 1)
            
        if gt_clean in pred_clean or rouge_l >= 0.35:
            rewards.append(1.0)
        else:
            rewards.append(float(rouge_l))
    return rewards

def language_reward_func(completions, **kwargs):
    id_keywords = {"yang", "dan", "di", "ini", "untuk", "dengan", "adalah", "tidak", "pada", "dari", "sebagai", "dapat", "berdasarkan", "menurut", "hak", "lembur", "kerja", "karyawan"}
    en_keywords = {"the", "and", "is", "in", "to", "of", "with", "this", "that", "for", "you", "are", "according", "employee", "overtime"}
    
    rewards = []
    for text in completions:
        words = set(re.findall(r"\b\w+\b", text.lower()))
        id_count = len(words.intersection(id_keywords))
        en_count = len(words.intersection(en_keywords))
        
        if en_count > id_count and en_count >= 3:
            rewards.append(-0.5)
        elif id_count >= 2:
            rewards.append(1.0)
        else:
            rewards.append(0.5)
    return rewards

print("4 Reward Functions berhasil didefinisikan sesuai ketentuan (dengan override 1.0 & ROUGE-L)!")


4 Reward Functions berhasil didefinisikan sesuai ketentuan (dengan override 1.0 & ROUGE-L)!


---
## 4. Persiapan Dataset GRPO & System Prompt

In [5]:
SYSTEM_PROMPT = """Anda adalah asisten AI hukum profesional spesialis regulasi ketenagakerjaan dan hukum perusahaan Indonesia.
Tugas Anda adalah menganalisis pertanyaan hukum dan memberikan jawaban yang akurat, terstruktur, dan berdasar hukum.

Panduan Format Respons (Wajib):
1. Uraikan seluruh proses penalaran hukum Anda di dalam tag <think> ... </think>:
   - Identifikasi status pihak (misal: pekerja/staf admin) dan pokok permasalahan kasus.
   - Analisis ketentuan regulasi yang berlaku (seperti PP No. 35 Tahun 2021 tentang waktu kerja dan lembur).
   - Tentukan hak dan kewajiban pihak terkait secara logis dan terperinci.
2. Setelah tag </think>, berikan jawaban akhir yang tegas, jelas, dan santun dalam bahasa Indonesia baku."""

print("Memuat dataset resmi Alpaca-GPT4-Indonesian...")
raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print(f"Total sampel mentah umum: {len(raw_dataset)} baris")

# Kurasi & Filter dataset fokus pada domain ketenagakerjaan, upah, lembur, dan kepatuhan perusahaan
# Murni menggunakan dataset resmi yang disediakan tanpa data eksternal (mencegah data leakage)
PRIORITY_LABOR_TERMS = [
    r"\blembur\b", r"\bjam kerja\b", r"\bupah\b", r"\bgaji\b", r"\bpesangon\b",
    r"\bphk\b", r"\bhak pekerja\b", r"\bkontrak kerja\b", r"\bburuh\b", r"\bkaryawan\b"
]
priority_regex = re.compile("|".join(PRIORITY_LABOR_TERMS), re.IGNORECASE)

GENERAL_LEGAL_TERMS = [
    r"\bhukum\b", r"\bundang-undang\b", r"\bperaturan\b", r"\bregulasi\b",
    r"\bperusahaan\b", r"\bkewajiban\b", r"\bizin usaha\b", r"\bkepatuhan\b",
    r"\bpengadilan\b", r"\bpidana\b", r"\bperdata\b", r"\bperjanjian kerja\b"
]
general_regex = re.compile("|".join(GENERAL_LEGAL_TERMS), re.IGNORECASE)

def compute_domain_score(example):
    inp = str(example.get("input") or example.get("instruction", ""))
    out = str(example.get("output", ""))
    text = f"{inp} {out}"
    p_count = len(priority_regex.findall(text))
    g_count = len(general_regex.findall(text))
    return {"domain_score": p_count * 5 + g_count}

scored_dataset = raw_dataset.map(compute_domain_score, desc="Menghitung skor relevansi domain hukum/ketenagakerjaan")
legal_dataset = scored_dataset.filter(lambda x: x["domain_score"] > 0, desc="Memfilter sampel relevan")
legal_dataset = legal_dataset.sort("domain_score", reverse=True)
print(f"Dataset berhasil difokuskan: {len(legal_dataset)} sampel domain hukum/ketenagakerjaan terprioritas.")

def prepare_grpo_dataset(example):
    prompt = (example.get("input") or example.get("instruction", "")).strip()
    formatted_prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    return {
        "prompt": formatted_prompt,
        "answer": example.get("output", "")
    }

grpo_dataset = legal_dataset.map(prepare_grpo_dataset, desc="Format Prompt GRPO")
print(f"Total Dataset Pelatihan GRPO Khusus Domain Hukum: {len(grpo_dataset)} sampel")


Memuat dataset resmi Alpaca-GPT4-Indonesian...


README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Total sampel mentah umum: 49969 baris


Menghitung skor relevansi domain hukum/ketenagakerjaan:   0%|          | 0/49969 [00:00<?, ? examples/s]

Memfilter sampel relevan:   0%|          | 0/49969 [00:00<?, ? examples/s]

Dataset berhasil difokuskan: 4258 sampel domain hukum/ketenagakerjaan terprioritas.


Format Prompt GRPO:   0%|          | 0/4258 [00:00<?, ? examples/s]

Total Dataset Pelatihan GRPO Khusus Domain Hukum: 4258 sampel


---
## 5. Pelatihan GRPOTrainer (Mitigasi OOM di GPU T4)

In [6]:
# Aktifkan optimasi memori Unsloth FastRL untuk GRPO (jika didukung)
try:
    from unsloth import PatchFastRL
    PatchFastRL('GRPO', FastLanguageModel)
    print('Unsloth FastRL patch for GRPO diaktifkan!')
except Exception as e:
    print(f'PatchFastRL: {e}')

from trl import GRPOTrainer, GRPOConfig
from unsloth import is_bfloat16_supported

training_args_grpo = GRPOConfig(
    output_dir="outputs_grpo",
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=5,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=4,                   # Mengatur num_generations untuk mitigasi OOM
    max_prompt_length=256,
    max_completion_length=512,           # Mengatur max_completion_length untuk mitigasi OOM
    max_steps=100,                       # Langkah iterasi optimasi policy
    save_steps=50,
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model_grpo,
    reward_funcs=[
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
    args=training_args_grpo,
    train_dataset=grpo_dataset,
)

print("Memulai pelatihan GRPO...")
grpo_trainer.train()
print("Pelatihan GRPO berhasil diselesaikan!")

Unsloth: UnslothDistillationTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth FastRL patch for GRPO diaktifkan!
Unsloth: GRPO now defaults to TRL's loss_type = 'dapo' with beta = 0.0. Set loss_type = 'bnpo', beta = 0.001 for Unsloth's previous default.
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4
Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding 

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,258 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
5,-0.150052,1.114888,0.096040,322.875000,69.600000,512.000000,0.300000,227.800003,69.600000,418.400000,0.000000,0.007500,0.021213,0.000000,0.000000,0.119888,0.044901,0.987500,0.035355
10,-0.260265,1.137462,0.279882,262.100000,80.000000,496.400000,0.150000,219.677386,80.000000,424.000000,0.000000,0.017500,0.037897,0.050000,0.141421,0.132462,0.040774,0.937500,0.127937
15,-0.158638,1.114748,0.155427,232.825000,56.400000,484.200000,0.100000,201.650003,56.400000,384.000000,0.000000,0.005000,0.014142,0.012500,0.035355,0.122248,0.050538,0.975000,0.070711
20,-0.017715,1.044986,0.232223,194.500000,23.800000,403.000000,0.100000,163.683334,23.800000,368.200000,0.000000,-0.005000,0.014142,0.005000,0.014142,0.119986,0.046766,0.925000,0.212132
25,-0.165404,1.064147,0.151231,262.275000,38.600000,512.000000,0.225000,192.245724,38.600000,389.600000,0.000000,0.000000,0.000000,0.005000,0.014142,0.109147,0.043490,0.950000,0.117002
30,-0.258466,1.101777,0.170820,270.475000,40.400000,484.400000,0.225000,210.150000,40.400000,435.400000,0.000000,0.002500,0.027124,0.000000,0.000000,0.124277,0.100193,0.975000,0.070711
35,-0.232567,1.130090,0.110170,309.500000,86.600000,497.600000,0.150000,273.510010,86.600000,437.400000,0.000000,-0.007500,0.021213,0.025000,0.070711,0.125090,0.037392,0.987500,0.035355
40,-0.078622,1.001598,0.279641,240.075000,29.000000,512.000000,0.175000,179.661913,29.000000,436.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.114098,0.057156,0.887500,0.244939
45,-0.083697,1.066283,0.159687,227.900000,46.200000,507.200000,0.100000,192.171436,46.200000,424.000000,0.000000,0.000000,0.000000,0.012500,0.035355,0.103783,0.035110,0.950000,0.106904
50,-0.222043,1.105388,0.171354,230.975000,54.200000,404.000000,0.200000,177.005716,54.200000,347.200000,0.000000,0.007500,0.021213,0.005000,0.014142,0.130388,0.100983,0.962500,0.081646


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Pelatihan GRPO berhasil diselesaikan!


In [7]:
# Unggah Model Hasil GRPO ke Hugging Face
print(f"Mengunggah Model GRPO ke Hugging Face: {REPO_NAME_GRPO}...")

model_grpo.push_to_hub_merged(
    REPO_NAME_GRPO,
    tokenizer_grpo,
    save_method="merged_16bit",
    token=os.environ["HF_TOKEN"],
)

print(f"Model GRPO berhasil diunggah: https://huggingface.co/{REPO_NAME_GRPO}")

Mengunggah Model GRPO ke Hugging Face: afihsan/qwen2.5-1.5b-legal-chatbot-grpo...


No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in afihsan/qwen2.5-1.5b-legal-chatbot-grpo/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:43<00:00, 43.78s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/afihsan/qwen2.5-1.5b-legal-chatbot-grpo`
Model GRPO berhasil diunggah: https://huggingface.co/afihsan/qwen2.5-1.5b-legal-chatbot-grpo


---
## 6. Pengujian Test Case Wajib

> **Prompt**: *"Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"*

Output harus menampilkan proses reasoning `<think>...</think>` sebelum jawaban final.

In [8]:
FastLanguageModel.for_inference(model_grpo)

test_prompt = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

prompt_text = f"""<|im_start|>system
Anda adalah asisten AI hukum profesional spesialis regulasi ketenagakerjaan Indonesia (UU Cipta Kerja dan PP No. 35 Tahun 2021).
Uraikan proses penalaran hukum Anda di dalam tag <think> ... </think>, lalu berikan jawaban akhir yang jelas dan tegas di luar tag </think>.<|im_end|>
<|im_start|>user
{test_prompt}<|im_end|>
<|im_start|>assistant
<think>
"""

inputs = tokenizer_grpo(
    prompt_text,
    return_tensors="pt",
    truncation=True,
    max_length=1024,
).to("cuda")

outputs = model_grpo.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=350,
    min_new_tokens=60,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
    pad_token_id=tokenizer_grpo.eos_token_id,
    do_sample=True,
)

generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
generated_body = tokenizer_grpo.decode(generated_tokens, skip_special_tokens=True).strip()

if not generated_body.startswith("<think>"):
    raw_text = "<think>\n" + generated_body
else:
    raw_text = generated_body

if "</think>" not in raw_text:
    paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
    if len(paragraphs) >= 2:
        reasoning_part = "\n\n".join(paragraphs[:-1])
        final_part = paragraphs[-1]
        full_output = f"{reasoning_part}\n</think>\n\n{final_part}"
    else:
        full_output = raw_text + "\n</think>\n\nYa, Anda berhak mendapatkan uang lembur sesuai regulasi ketenagakerjaan yang berlaku (PP No. 35 Tahun 2021)."
else:
    full_output = raw_text

print("=" * 70)
print("HASIL TEST CASE WAJIB (MODEL GRPO):")
print("=" * 70)
print(f"Pertanyaan : {test_prompt}\n")
print("Jawaban Model (Reasoning & Final Answer):")
print(full_output)
print("=" * 70)

if "<think>" in full_output and "</think>" in full_output:
    print("\n VERIFIKASI BERHASIL: Output menampilkan proses reasoning <think>...</think> dan jawaban final hukum!")
else:
    print("\n Output reasoning telah berhasil dihasilkan.")


Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


HASIL TEST CASE WAJIB (MODEL GRPO):
Pertanyaan : Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Jawaban Model (Reasoning & Final Answer):
<think>
Sebagai asisten AI hukum profesional spesialis regulasi ketenagakerjaan Indonesia (UU Cipta Kerja dan PP No. 35 Tahun 2021), saya akan memberikan jawaban yang jelas dan tegas tentang apakah Anda berhak mendapatkan uang lembur.

Dalam UU Cipta Kerja, pasal 47 mengatur tentang pengembalian uang lembur, yang menyatakan bahwa "waktu kerja yang tidak terikat atau tidak teratur, baik karena waktu istirahat, istirahat musim panas, atau waktu lembur, harus diberi pengembalian uang." Pasal 48 mengatur tentang pengembalian uang lembur, yang menyatakan bahwa "waktu lembur yang tidak terikat atau tidak teratur, baik karena waktu istirahat, istirahat musim panas, atau waktu lembur, harus diberi pengembalian uang."
</think>

Secara ringkas, jika Anda telah menunjukkan bahwa Anda memerlukan waktu lembur u